# Vinograd, Nair et al. (2024): rSLDS fits to the miniscope and two-photon data

Paper: Vinograd A, Nair A, Kim JH, Linderman SW, Anderson DJ (2024). Causal evidence of a line
attractor encoding an affective state. Nature 634, 910-918.
[doi:10.1038/s41586-024-07915-x](https://doi.org/10.1038/s41586-024-07915-x) (version of record,
with Methods and Extended Data legends).

Dataset: [DANDI:001037](https://dandiarchive.org/dandiset/001037/0.260609.1402), version
0.260609.1402, the most recent published version and the only one holding all five miniscope mice
(see `stored_weights_and_stimulation.ipynb` for the files of each version and the table of all
claims).

The dandiset stores no rSLDS model for the miniscope sessions and only the factor weights for the
two-photon sessions, so the claims about time constants, line attractor scores and miniscope x1
neurons need new fits. The fits use the library the paper names (ssm, at commit eb6c8aa) with the
settings of the authors' sample code linked from the dandiset (claims header), in
`linattr.MINI_KW` (3 states) and `linattr.TWO_P_KW` (2 states), all with 2 latent dimensions.
The main reading uses seed 0; the sample code tries seeds 0, 1, 137 and 1337 and keeps the best by
its R2, which is shown as a variant. The Methods choose the number of states by fivefold
cross-validation; that reading is in `cross_validation.ipynb`.

Claims tested here: ed1n-engaging-x1-slower, ed1p-observing-x1-slower,
ed3c-x1-activity-observe-vs-engage, ed3e-achw-full-session, ed3f-achw-after-removal,
ed3g-decode-observed-attack, ed3h-decode-engaged-from-observed, ed4c-x1-neuron-overlap,
ed4d-weight-dot-product, ed4g-x2-more-spontaneous, ed4hi-amplitude-tau-similar,
fig1e-2p-x1-slower, fig1f-line-attractor-score, x1-time-constant-range, ed8c-model-integrates.
ed1m-x1-observe-attack-vs-sniff is blocked (data missing); the data check shows why.

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np

import linattr as L

warnings.filterwarnings("ignore")
started = time.time()
print(L.ensure_ssm())
MS = {s: L.load_miniscope(s) for s in L.MINISCOPE}
TP = {s: L.load_2p(s) for s in L.TWO_PHOTON}
print(f"dandiset {L.DANDISET} version {L.VERSION}: {len(MS)} miniscope and {len(TP)} two-photon mice loaded "
      f"in {time.time() - started:.0f} s")

## Data check

In [ ]:
for s, m in MS.items():
    meta = m["meta"]
    print(f"{s}: session order in file {meta['session_key']}; cells {meta['n_cells']}; frames {meta['n_frames']}; "
          f"rates {[m[c]['rate'] for c in ['engage', 'observe']]} Hz")
    for c in ["engage", "observe"]:
        Y = m[c]["Y"]
        first = min(v[0][0] for v in m[c]["beh"].values() if v)
        labels = {k: len(v) for k, v in m[c]["beh"].items()}
        print(f"    {c:8s} {Y.shape[0]} frames ({Y.shape[0] / 10 / 60:.1f} min) x {Y.shape[1]} cells; mean {Y.mean():.3f}, "
              f"sd {Y.std():.3f}, range {Y.min():.2f} to {Y.max():.2f}; first annotation at {first:.1f} s; "
              f"bouts per label {labels}")
print("\nMice with sniffing of the SW mouse annotated (ed1m):",
      [s for s, m in MS.items() if L.miniscope_labels(m["observe"], "watch_sniff")])

Each miniscope file holds both sessions with the same number of cells; that the cells are the same
in both sessions is not stated in the files and is assumed (the paper compares single-cell weights
across conditions, Extended Data Fig. 4a-d). The traces are z-scored (mean 0, SD near 1). The
sessions are 6.0 to 7.0 min long, and the first annotation comes 56 to 152 s after the start, so the
5 min solitary baseline the Methods describe at the start of every session is not in the files; the
baseline used here is the time before the first annotation. Sniffing of the SW mouse is annotated
in M30L only, while Extended Data Fig. 1m uses 4 mice, so ed1m-x1-observe-attack-vs-sniff is
blocked. M30L names its labels differently (`balb_c_in_cage`, with separate intruder entries); the
others use `balb_c` and `sw`.

Clock check: if annotations share the activity's clock, the population activity differs between
attack frames and other frames more with the true labels than with labels shifted by 60 s.

In [ ]:
for c, lab in [("engage", "attack"), ("observe", "watch_attack")]:
    for shift in [0, -60, 60]:
        v = []
        for s, m in MS.items():
            Y = m[c]["Y"]
            iv = [(a + shift, b + shift) for a, b in L.miniscope_labels(m[c], lab)]
            r = L.raster(iv, len(Y))
            pop = Y.mean(1)
            v.append(pop[r].mean() - pop[~r].mean() if r.any() else np.nan)
        print(f"{c:8s} {lab:12s} labels shifted {shift:+3d} s: population z during bouts minus elsewhere, "
              f"per mouse {np.round(v, 2).tolist()}")

In the engaging sessions the population is more active during attack with the true labels (0.10 to
0.30 z) than with labels shifted 60 s either way, in every mouse but M38R (0.19 with labels 60 s
early). In the observation sessions the contrast against labels shifted 60 s later is clear, but
labels shifted 60 s earlier give a similar contrast in three mice, as activity stays raised after
the intruder enters. The annotations are consistent with the activity's clock; for the observation
sessions this is not shown beyond that.

## The fits

One fit per session and seed, in parallel processes. Two-photon: the observation session of each
mouse, autonomous, and for M70, the only mouse with BALB/c intervals, also with the BALB/c-presence
input (Methods: "Flow fields obtained from head-fixed animals observing aggression where fit with
input terms representing the presence of the BALB/c intruder").

In [ ]:
jobs = []
for seed in L.CODE_SEEDS:
    for s, m in MS.items():
        for c in ["engage", "observe"]:
            jobs.append((("mini", s, c, seed), m[c]["Y"], dict(L.MINI_KW, seed=seed), None))
    for s, d in TP.items():
        jobs.append((("2p", s, "auto", seed), d["sessions"]["observation"], dict(L.TWO_P_KW, seed=seed), None))
    Y70 = TP["M70"]["sessions"]["observation"]
    u70 = L.raster(TP["M70"]["beh"]["balbc_in_cage"], len(Y70)).astype(float)[:, None]
    jobs.append((("2p", "M70", "input", seed), Y70, dict(L.TWO_P_KW, seed=seed), u70))
t0 = time.time()
FITS = L.run_fits(jobs)
print(f"{len(FITS)} fits in {(time.time() - t0) / 60:.1f} min; per fit {np.median([f['seconds'] for f in FITS.values()]):.0f} s (median)")


def best_seed(kind, s, c):
    keys = [(kind, s, c, seed) for seed in L.CODE_SEEDS]
    return max(keys, key=lambda k: FITS[k]["code_r2"])


def fit_of(kind, s, c, reading):
    return FITS[(kind, s, c, 0)] if reading == "seed0" else FITS[best_seed(kind, s, c)]


print("Sample-code R2 (latent forward prediction) per seed, and time constants (s) of every state:")
for kind, s, c in sorted({k[:3] for k in FITS}):
    r2 = [FITS[(kind, s, c, seed)]["code_r2"] for seed in L.CODE_SEEDS]
    sf = L.slow_fast(FITS[(kind, s, c, 0)])
    print(f"  {kind:4s} {s:6s} {c:7s} R2 by seed {np.round(r2, 2).tolist()} (best seed {best_seed(kind, s, c)[3]}); "
          f"seed 0 states: {[np.round(t, 1).tolist() for t in sf['taus']]}")

## ed1n-engaging-x1-slower and ed1p-observing-x1-slower

> In freely behaving mice engaging in aggression, rSLDS fits show an integration dimension x1 with a
> longer time constant than the orthogonal dimension x2, an approximate line attractor.
>
> In the same mice observing aggression, rSLDS fits also show an x1 dimension with a longer time
> constant than x2.

In the state with the largest time constant, x1 is the larger and x2 the smaller of its two time
constants (criterion). A state whose eigenvalues are a complex pair has one time constant for both.

In [ ]:
def tau_test(kind, mice, c, reading):
    rows = [L.slow_fast(fit_of(kind, s, c, reading)) for s in mice]
    x1 = np.array([r["tau_x1"] for r in rows]); x2 = np.array([r["tau_x2"] for r in rows])
    las = np.array([r["las"] for r in rows])
    p = L.mwu(x1, x2)
    print(f"{kind} {c} ({reading}): x1 {np.round(x1, 1).tolist()} s; x2 {np.round(x2, 1).tolist()} s; "
          f"line attractor score {np.round(las, 2).tolist()}")
    print(f"    x1 > x2 in {(x1 > x2).sum()} of {len(x1)}; equal (complex pair) in {(np.isclose(x1, x2)).sum()}; "
          f"Mann-Whitney p = {p:.4f}; mean score {las.mean():.2f}")
    return x1, x2, las, p

TAU = {}
for c in ["engage", "observe"]:
    for reading in ["seed0", "code_seed"]:
        TAU[(c, reading)] = tau_test("mini", L.MINISCOPE, c, reading)

**Result (ed1n-engaging-x1-slower): not reproduced.** In the engaging sessions, x1 is slower than x2
in 3 of 5 mice with seed 0; in the other two the slowest state has a complex pair of eigenvalues, so
x1 and x2 share one time constant (line attractor score 0). The Mann-Whitney test gives p = 0.093
(paper 0.0079); with the seeds the sample code would keep, x1 is slower in 2 of 5 (p = 0.75). Where
x1 is slower its time constant is 95 to 740 s and x2's 7 to 39 s.

**Result (ed1p-observing-x1-slower): not reproduced.** In the observation sessions x1 is slower in 3
of 5 mice under both seed readings (p = 0.21 with seed 0, 0.40 with the code's seeds); the other two
have complex slow states. The number of states chosen by cross-validation is the other reading, in
`cross_validation.ipynb`.

## x1 and x2 neurons of the miniscope fits

Each neuron's weight on x1 and x2 is its row of the emission matrix C mapped onto the slow and the
other eigenvector of the slow state (Methods: the columns of C identify the neurons contributing to
each dimension); for a complex pair the columns of C are used. x1 neurons are the five with the
largest absolute x1 weight (criterion, after Fig. 1i).

In [ ]:
WTS = {(s, c, r): L.x1_weights(fit_of("mini", s, c, r)) for s in L.MINISCOPE for c in ["engage", "observe"]
       for r in ["seed0", "code_seed"]}
for s in L.MINISCOPE:
    print(s, {c: sorted(L.top_neurons(WTS[(s, c, 'seed0')][0]).tolist()) for c in ["engage", "observe"]})

## ed3c-x1-activity-observe-vs-engage

> The activity of x1 neurons during observed attack bouts does not differ from their activity during
> engaged attack.

The dandiset marks no x1 neurons for the miniscope mice, so the only reading is the five neurons
with the largest x1 weight of each condition's fit.

In [ ]:
for reading in ["seed0", "code_seed"]:
    oa, ea = [], []
    for s, m in MS.items():
        for c, lab, store in [("observe", "watch_attack", oa), ("engage", "attack", ea)]:
            Y = m[c]["Y"]; r = L.raster(L.miniscope_labels(m[c], lab), len(Y))
            store.append(Y[r][:, L.top_neurons(WTS[(s, c, reading)][0])].mean())
    print(f"{reading}: observed attack {np.round(oa, 2).tolist()}, engaged attack {np.round(ea, 2).tolist()}; "
          f"mean difference {np.mean(oa) - np.mean(ea):+.2f} z; Mann-Whitney p = {L.mwu(oa, ea):.3f}")

**Result: reproduced.** The x1 neurons' mean activity during observed attack (0.36 z, seed 0) and
engaged attack (0.31 z) does not differ by the paper's test (p = 1.0; with the code's seeds 0.84,
difference +0.06 z).

## ed3e-achw-full-session and ed3f-achw-after-removal

> Over the full interaction, autocorrelation half-widths of x1 neurons are similar when observing
> (25 +/- 0.8 s) and engaging in aggression (20 +/- 1.7 s).
>
> Aligned to removal of the last intruder, autocorrelation half-widths of x1 neurons are similar when
> observing (14 +/- 1 s) and engaging (11 +/- 1.6 s).

Windows: whole session, or first BALB/c entry to last BALB/c removal (ed3e); last removal to the end
of the session (ed3f).

In [ ]:
def span(sess):
    iv = L.miniscope_labels(sess, "balbc")
    return int(round(iv[0][0] * L.RATE)), int(round(iv[-1][1] * L.RATE))

ACHW = {}
for reading in ["seed0", "code_seed"]:
    for window in ["whole", "interaction", "after"]:
        vals = {}
        for c in ["observe", "engage"]:
            v = []
            for s, m in MS.items():
                Y = m[c]["Y"]; a, b = span(m[c])
                seg = {"whole": Y, "interaction": Y[a:b], "after": Y[b:]}[window]
                v.append(np.nanmean([L.achw(seg[:, j]) for j in L.top_neurons(WTS[(s, c, reading)][0])]))
            vals[c] = np.array(v)
        o, e = vals["observe"], vals["engage"]
        ACHW[(reading, window)] = (o, e)
        print(f"{reading:9s} {window:11s}: observe {np.round(o, 1).tolist()} mean {o.mean():.1f} +/- {o.std(ddof=1) / np.sqrt(5):.1f}; "
              f"engage {np.round(e, 1).tolist()} mean {e.mean():.1f} +/- {e.std(ddof=1) / np.sqrt(5):.1f}; "
              f"Mann-Whitney p = {L.mwu(o, e):.3f}, signed-rank p = {L.wsr(o, e):.3f}")
print("Window lengths after the last removal (s):", {s: round((len(m['observe']['Y']) - span(m['observe'])[1]) / 10)
                                                     for s, m in MS.items()}, "(observe)",
      {s: round((len(m['engage']['Y']) - span(m['engage'])[1]) / 10) for s, m in MS.items()}, "(engage)")

**Result (ed3e-achw-full-session): not reproduced.** Over the whole session the x1 half-widths are
40.8 s when observing and 27.0 s when engaging (seed 0; expected 25 +/- 0.8 and 20 +/- 1.7), and the
unpaired test finds them different (p = 0.016; signed-rank p = 0.062). From first entry to last
removal they are 9.9 and 14.9 s (p = 0.31 and 0.31). No reading puts the values within tolerance;
the code's seeds give the same picture (38.6 and 23.6 s; 10.2 and 13.7 s).

**Result (ed3f-achw-after-removal): partially reproduced.** After the last removal the half-widths are
similar in the two conditions, as the paper finds (12.7 against 8.7 s, Mann-Whitney p = 0.095,
signed-rank p = 0.125, seed 0; code's seeds 9.2 against 7.6 s, p = 0.55 and 0.81), and the
observation value is longer, as in the paper, but neither value is within tolerance (14 +/- 1 and
11 +/- 1.6 s). The windows after removal are 71 to 147 s long.

## ed3g-decode-observed-attack and ed3h-decode-engaged-from-observed

> A decoder trained on x1 activity during observation separates observed attack bouts from
> non-attack bouts better than shuffled controls.
>
> The decoder trained on observing attack decodes engaged attack from x1 activity better than
> shuffled data.

The x1 signal is the z-scored activity projected on the unit x1 weight vector, then z-scored over the
session ("normalized", Methods). Trials: observed attack bouts merged when less than 5 s apart, each
with as many non-attack frames drawn at random (balancing). Shuffles assign the labels of each
trial's two bouts at random, 1,000 times per mouse (Methods). For ed3h, the engaging session is
projected on the observation model's x1 weights, or on its own (sign chosen to agree with the
observation weights).

In [ ]:
rng = np.random.default_rng(3)
N_SHUF = 1000

def x1_signal(Y, w):
    sig = Y @ w
    return (sig - sig.mean()) / sig.std()

DEC = {}
for reading in ["seed0", "code_seed"]:
    TR = {}
    for s, m in MS.items():
        TR[s] = L.decoder_trials(x1_signal(m["observe"]["Y"], WTS[(s, "observe", reading)][0]),
                                 L.miniscope_labels(m["observe"], "watch_attack"), rng)
    for split in ["one", "loto", "5fold"]:
        real = [L.decode(TR[s], split) for s in MS]
        shuf = [np.mean([L.decode(L.shuffle_trials(TR[s], rng), split) for _ in range(N_SHUF)]) for s in MS]
        DEC[(reading, split)] = (real, shuf)
        print(f"{reading:9s} split {split:5s}: trials per mouse {[len(TR[s]) for s in MS]}; accuracy {np.round(real, 2).tolist()}, "
              f"shuffled {np.round(shuf, 2).tolist()}; Mann-Whitney p = {L.mwu(real, shuf):.4f}")

**Result (ed3g-decode-observed-attack): reproduced, with the seed-0 fits.** With seed 0 the decoder
beats the shuffled decoder in every mouse under all three splits (accuracy 0.66 to 0.83 against 0.49
to 0.51), Mann-Whitney p = 0.0079, the smallest possible with 5 mice against 5. With the code's
seeds the same holds for the leave-one-out and fivefold splits (p = 0.0079), but training on one
trial gives p = 0.15, as M35R's decoder is at chance (0.48).

In [ ]:
rng = np.random.default_rng(4)
for reading in ["seed0", "code_seed"]:
    for weights in ["observation", "own"]:
        real, shuf = [], []
        for s, m in MS.items():
            wo = WTS[(s, "observe", reading)][0]
            we = wo if weights == "observation" else WTS[(s, "engage", reading)][0] * np.sign(WTS[(s, "engage", reading)][0] @ wo)
            tr_o = L.decoder_trials(x1_signal(m["observe"]["Y"], wo), L.miniscope_labels(m["observe"], "watch_attack"), rng)
            tr_e = L.decoder_trials(x1_signal(m["engage"]["Y"], we), L.miniscope_labels(m["engage"], "attack"), rng)
            clf = L.fit_threshold(np.concatenate([a for a, b in tr_o]), np.concatenate([b for a, b in tr_o]))
            real.append(np.mean([L.accuracy(clf, t) for t in tr_e]))
            sh = []
            for _ in range(N_SHUF):
                so = L.shuffle_trials(tr_o, rng)
                c2 = L.fit_threshold(np.concatenate([a for a, b in so]), np.concatenate([b for a, b in so]))
                sh.append(np.mean([L.accuracy(c2, t) for t in tr_e]))
            shuf.append(np.mean(sh))
        print(f"{reading:9s} engaging projected on {weights:11s} weights: accuracy {np.round(real, 2).tolist()}, "
              f"shuffled {np.round(shuf, 2).tolist()}; Mann-Whitney p = {L.mwu(real, shuf):.4f}")

**Result (ed3h-decode-engaged-from-observed): reproduced, with the seed-0 fits.** With seed 0 the
observation decoder classifies engaged attack better than the shuffled decoders in every mouse,
whether the engaging session is projected on the observation weights (0.60 to 0.86 against 0.49 to
0.52) or on its own (0.59 to 0.92), p = 0.0079 both. With the code's seeds one mouse falls below
chance and p = 0.15 under both projections.

## ed4c-x1-neuron-overlap and ed4d-weight-dot-product

> Around 70% of the neurons that contribute to the line attractor overlap between observing and
> engaging in aggression.
>
> The x1 weight vector during observation is more aligned with the x1 weight vector during
> engagement than with the x2 weight vector.

Contributing neurons: the five with the largest absolute x1 weight of each fit (the dandiset marks
none for the miniscope mice). With five in each set the two denominators are the same.

In [ ]:
for reading in ["seed0", "code_seed"]:
    ov, chance = [], []
    for s in L.MINISCOPE:
        A = set(L.top_neurons(WTS[(s, "engage", reading)][0])); B = set(L.top_neurons(WTS[(s, "observe", reading)][0]))
        ov.append(len(A & B) / 5); chance.append(5 / MS[s]["engage"]["Y"].shape[1])
    print(f"{reading}: overlap per mouse {np.round(ov, 2).tolist()}, mean {100 * np.mean(ov):.0f}% "
          f"(expected 70 +/- 7%); chance (5 of N cells) {100 * np.mean(chance):.0f}%")
    for n in [10, 20]:
        ovn = [len(set(L.top_neurons(WTS[(s, 'engage', reading)][0], n)) & set(L.top_neurons(WTS[(s, 'observe', reading)][0], n))) / n
               for s in L.MINISCOPE]
        print(f"     top {n} instead of 5 (variant): mean {100 * np.mean(ovn):.0f}%")

**Result (ed4c-x1-neuron-overlap): not reproduced.** The five strongest x1 neurons of the engaging and
observation fits overlap by 4% on average under both seed readings (one shared neuron in M30L, none
elsewhere), against 70 +/- 7% expected and 7% by chance; the top 10 and top 20 overlap by 16 to 34%.

In [ ]:
for reading in ["seed0", "code_seed"]:
    d11, d12 = [], []
    for s in L.MINISCOPE:
        e1, e2 = WTS[(s, "engage", reading)]; o1, _ = WTS[(s, "observe", reading)]
        d11.append(o1 @ e1); d12.append(o1 @ e2)
    d11, d12 = np.array(d11), np.array(d12)
    print(f"{reading}: raw x1.x1 {np.round(d11, 2).tolist()} x1.x2 {np.round(d12, 2).tolist()} p = {L.mwu(d11, d12):.4f}; "
          f"absolute x1.x1 {np.round(abs(d11), 2).tolist()} x1.x2 {np.round(abs(d12), 2).tolist()} p = {L.mwu(abs(d11), abs(d12)):.4f}; "
          f"x1.x1 larger (absolute) in {(abs(d11) > abs(d12)).sum()} of 5")

**Result (ed4d-weight-dot-product): not reproduced.** With seed 0 the observation x1 vector is more
aligned with the engagement x1 vector than with x2 in 3 of 5 mice (absolute dot products 0.15 to
0.57 against 0.11 to 0.51), Mann-Whitney p = 0.22 for raw and absolute products; with the code's
seeds in 2 of 5 (p = 0.55 and 0.69).

## ed4g-x2-more-spontaneous and ed4hi-amplitude-tau-similar

> During the solitary baseline, x2 neurons show more frequent spontaneous calcium transients than x1
> neurons (2.3 against 1.6 events).
>
> Baseline transient amplitude (0.58 against 0.71 z) and decay constant (1.7 against 2.3 s) do not
> differ between x1 and x2 neurons.

Baseline: from the start of the session to the first annotation (the files hold no 5 min baseline).
A transient is an upward crossing of 1.5 SD of the activity z-scored per neuron over the session
(legend of Extended Data Fig. 4g). Readings: frequency per baseline and per minute; the engaging and
the observation session, each with its own model's x1 and x2 neurons. Amplitude: the peak z of each
transient; decay constant: exponential fit to each mouse's mean transient from its peak.

In [ ]:
TRANS = {}
for reading in ["seed0", "code_seed"]:
    for c in ["engage", "observe"]:
        rows = []
        for s, m in MS.items():
            Y = m[c]["Y"]; Y = (Y - Y.mean(0)) / Y.std(0)
            first = min(v[0][0] for v in m[c]["beh"].values() if v)
            B = Y[:int(first * L.RATE)]
            w1, w2 = WTS[(s, c, reading)]
            out = {"baseline_min": len(B) / L.RATE / 60}
            for name, w in [("x1", w1), ("x2", w2)]:
                cnt, amp, pk = [], [], []
                for j in L.top_neurons(w):
                    on, peaks, idx = L.transients(B[:, j])
                    cnt.append(len(on)); amp += peaks.tolist(); pk.append((j, idx))
                out[name + "_n"] = np.mean(cnt)
                out[name + "_amp"] = np.mean(amp) if amp else np.nan
                segs = [L.decay_tau(B[:, j], idx) for j, idx in pk]
                out[name + "_tau"] = np.nanmean(segs) if np.isfinite(segs).any() else np.nan
            rows.append(out)
        TRANS[(reading, c)] = rows
        bm = np.array([r["baseline_min"] for r in rows])
        n1 = np.array([r["x1_n"] for r in rows]); n2 = np.array([r["x2_n"] for r in rows])
        print(f"{reading:9s} {c:7s}: baseline {np.round(bm, 2).tolist()} min")
        print(f"     events per baseline x1 {np.round(n1, 2).tolist()} mean {n1.mean():.2f}; x2 {np.round(n2, 2).tolist()} mean {n2.mean():.2f}; "
              f"Mann-Whitney p = {L.mwu(n1, n2):.3f}; x2 > x1 in {(n2 > n1).sum()} of 5")
        print(f"     events per minute x1 mean {(n1 / bm).mean():.2f}; x2 mean {(n2 / bm).mean():.2f}; Mann-Whitney p = {L.mwu(n1 / bm, n2 / bm):.3f}")

**Result (ed4g-x2-more-spontaneous): not reproduced.** No reading finds more transients in x2 than in
x1 neurons: with seed 0, 1.68 against 3.12 events per baseline in the engaging sessions (p = 0.46,
x2 higher in 3 of 5 mice) and 1.72 against 1.12 in the observation sessions (p = 1.0); per minute
and with the code's seeds p is 0.40 to 1.0. The baselines are 0.94 to 1.60 min, not 5 min.

In [ ]:
for reading in ["seed0", "code_seed"]:
    for c in ["engage", "observe"]:
        rows = TRANS[(reading, c)]
        for meas, unit in [("amp", "z"), ("tau", "s")]:
            a = np.array([r["x1_" + meas] for r in rows]); b = np.array([r["x2_" + meas] for r in rows])
            ok = np.isfinite(a) & np.isfinite(b)
            print(f"{reading:9s} {c:7s} {meas}: x1 {np.round(a, 2).tolist()} mean {np.nanmean(a):.2f} {unit}; "
                  f"x2 {np.round(b, 2).tolist()} mean {np.nanmean(b):.2f}; mice with both {ok.sum()}; "
                  f"Mann-Whitney p = {L.mwu(a[ok], b[ok]):.3f}; signed-rank p = {L.wsr(a[ok], b[ok]) if ok.sum() > 1 else float('nan'):.3f}")

**Result (ed4hi-amplitude-tau-similar): partially reproduced.** No reading finds a difference
between x1 and x2 neurons in amplitude or decay constant (all p >= 0.125), as the paper finds, but the
values are not those the paper reports: peak amplitudes are 1.5 to 2.3 z (expected 0.58 and 0.71 z,
which are below the 1.5 SD detection threshold, so the paper must measure amplitude another way)
and decay constants range from 0.05 to the fit's bound of 100 s. Several mice have no transient in
one set within the short baseline, so some tests use 3 or 4 mice.

## fig1e-2p-x1-slower, fig1f-line-attractor-score and x1-time-constant-range

> In head-fixed mice observing aggression, rSLDS fits to 2P data show an x1 dimension with longer
> time constants than x2.
>
> The line attractor score is positive across the 9 head-fixed mice.
>
> The experimentally observed network time constants of the integration dimension lie in a range
> of about 50 to 200 s.

In [ ]:
TAU2 = {}
for reading in ["seed0", "code_seed"]:
    TAU2[reading] = tau_test("2p", L.TWO_PHOTON, "auto", reading)
for reading in ["seed0", "code_seed"]:
    sf = L.slow_fast(fit_of("2p", "M70", "input", reading))
    print(f"M70 with the BALB/c input ({reading}): x1 {sf['tau_x1']:.1f} s, x2 {sf['tau_x2']:.1f} s, score {sf['las']:.2f}; "
          f"states {[np.round(t, 1).tolist() for t in sf['taus']]}")
for reading in ["seed0", "code_seed"]:
    las = TAU2[reading][2]
    print(f"fig1f ({reading}): scores {np.round(las, 2).tolist()}; mean {las.mean():.2f}; above 0 in {(las > 0).sum()} of {len(las)}")

**Result (fig1e-2p-x1-slower): not reproduced.** x1 is slower than x2 in 5 of 7 mice; in M70 and M18R
the slowest state has a complex pair (63 and 137 s for both dimensions). The Mann-Whitney test gives
p = 0.37 with seed 0 and 0.096 with the code's seeds (paper p < 0.0001 over 9 mice; with 7 mice
against 7 the smallest possible p is 0.0006). M70 fit with the BALB/c input has x1 30,479 s and x2
2.9 s; the other six mice have no BALB/c intervals, so that reading covers one mouse.

**Result (fig1f-line-attractor-score): partially reproduced.** The mean score is positive (0.83 with
seed 0, 1.32 with the code's seeds), but the score is above 0 in 5 of 7 mice only; M70 and M18R
score 0.

In [ ]:
for reading in ["seed0", "code_seed"]:
    x1_2p = TAU2[reading][0]
    x1_all = np.r_[x1_2p, TAU[("engage", reading)][0], TAU[("observe", reading)][0]]
    for name, v in [("2P observation fits (7)", x1_2p), ("2P and miniscope fits (17)", x1_all)]:
        inside = (v >= 45) & (v <= 220)
        print(f"{reading:9s} {name}: x1 time constants {np.round(np.sort(v), 0).tolist()} s; inside 45-220 s: "
              f"{inside.sum()} of {len(v)}; median {np.median(v):.0f} s")

**Result (x1-time-constant-range): partially reproduced.** The median x1 time constant of the 2P fits
is 114 s, inside 45 to 220 s, but 2 of 7 fits are outside (474 and 2,984 s with seed 0; 274 and 474
s with the code's seeds). With the miniscope fits added, 12 of 17 (13 of 17) are inside, median 137 s
(121 s).

## ed8c-model-integrates

> Given the same pulsed input, the rSLDS model fit to observation data also integrates along x1, as in
> the data.

Each mouse's observation model (seed 0) is simulated forward without noise from the latent state of
the activation session's baseline (the baseline activity mapped to the latents through the
pseudo-inverse of C), with discrete states chosen by the model's recurrent transitions. Reading (b):
at the end of each 2 s pulse the latent state is moved along the slow eigenvector by the size of the
first evoked x1 response in the data (latent x1 at onset + 2 s minus baseline). Reading (a): the M70
model fit with the BALB/c input, driven by that input for 2 s at each onset. Simulated x1 is read at
the end of each ISI.

In [ ]:
def simulate(fit, x0, T, onsets, kick=None, u=None):
    x = np.zeros((T, x0.size)); x[0] = x0
    for t in range(T - 1):
        logits = fit["Rs"] @ x[t] + fit["r"]
        k = int(np.argmax(logits))
        nxt = fit["As"][k] @ x[t] + fit["bs"][k]
        if u is not None:
            nxt = nxt + fit["Vs"][k] @ u[t]
        if kick is not None and (t + 1 - L.PULSE_FRAMES) in onsets:
            nxt = nxt + kick
        x[t + 1] = nxt
    return x

def isi_ends(tr, on):
    isi = on[1] - on[0]
    return np.array([tr[min(on[i + 1] - 1 if i + 1 < len(on) else o + isi, len(tr) - 1)] for i, o in enumerate(on)])

ses = "activation_integration_neurons"
rises, r2s = [], []
for s, d in TP.items():
    fit = FITS[("2p", s, "auto", 0)]
    sf = L.slow_fast(fit); v1 = sf["v1"]
    Cp = np.linalg.pinv(fit["C"])
    Y = d["sessions"][ses]; on = list(d["stim"][ses])
    xd = (Y - fit["d"]) @ Cp.T
    x0 = xd[:on[0]].mean(0)
    size = (xd[on[0] + L.PULSE_FRAMES] - x0) @ v1
    if size < 0:
        v1, size = -v1, -size
    sim = simulate(fit, x0, len(Y), on, kick=size * v1)
    s1 = (sim - x0) @ v1; d1 = (xd - x0) @ v1
    ends = isi_ends(s1, on); dends = isi_ends(d1, on)
    r2 = 1 - np.sum((d1 - s1) ** 2) / np.sum((d1 - d1.mean()) ** 2)
    rises.append(ends[2] > ends[0]); r2s.append(r2)
    print(f"{s}: kick {size:.2f}; simulated x1 at ISI ends {np.round(ends, 2).tolist()}; data {np.round(dends, 2).tolist()}; "
          f"rises ISI1 -> ISI3: {ends[2] > ends[0]}; R2 simulated against data x1 {r2:.2f}")
print(f"(b) rises in {sum(rises)} of {len(rises)} mice; R2 median {np.median(r2s):.2f}")
fit = FITS[("2p", "M70", "input", 0)]
d = TP["M70"]; Y = d["sessions"][ses]; on = list(d["stim"][ses])
u = np.zeros((len(Y), 1))
for o in on:
    u[o:o + L.PULSE_FRAMES] = 1
Cp = np.linalg.pinv(fit["C"]); xd = (Y - fit["d"]) @ Cp.T; x0 = xd[:on[0]].mean(0)
v1 = L.slow_fast(fit)["v1"]
sim = simulate(fit, x0, len(Y), on, u=u)
s1 = (sim - x0) @ v1
print(f"(a) M70 input model: simulated x1 at ISI ends {np.round(isi_ends(s1, on), 2).tolist()}")

**Result (ed8c-model-integrates): reproduced, under reading (b).** With a kick of the size of the
first evoked response at each pulse, the simulated x1 at the end of the ISI rises from ISI1 to ISI3
in 7 of 7 mice. The simulations match the data poorly (R2 of simulated against data x1 from -14.3
to 0.14, median -1.32; the paper reports about 85%), and under reading (a), possible only for M70,
the input-driven model's x1 falls (-0.49 to -1.28). The rise under (b) follows from adding a pulse
along a slowly decaying dimension, which is what the claim tests.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| ed1n-engaging-x1-slower | not reproduced |
| ed1p-observing-x1-slower | not reproduced |
| ed3c-x1-activity-observe-vs-engage | reproduced |
| ed3e-achw-full-session | not reproduced |
| ed3f-achw-after-removal | partially reproduced |
| ed3g-decode-observed-attack | reproduced (assumed: seed-0 fits) |
| ed3h-decode-engaged-from-observed | reproduced (assumed: seed-0 fits) |
| ed4c-x1-neuron-overlap | not reproduced |
| ed4d-weight-dot-product | not reproduced |
| ed4g-x2-more-spontaneous | not reproduced |
| ed4hi-amplitude-tau-similar | partially reproduced |
| fig1e-2p-x1-slower | not reproduced |
| fig1f-line-attractor-score | partially reproduced |
| x1-time-constant-range | partially reproduced |
| ed8c-model-integrates | reproduced |
| ed1m-x1-observe-attack-vs-sniff | blocked: data missing |

How the analysis differs from the paper's: the models are new fits with the sample code's settings
(number of states, penalties, iterations) and seed 0, not the paper's models, which the dandiset
does not hold (only the two-photon factor weights); the cross-validated number of states is a second
reading in `cross_validation.ipynb`. x1 neurons of the miniscope fits are the five strongest by the
emission weights mapped on the slow eigenvector. The baseline is the time before the first
annotation, 0.9 to 1.6 min, not the 5 min the Methods describe.

What was found about the dataset: the miniscope files hold the traces and annotations of both
sessions of 5 mice but no rSLDS models, no x1 or x2 labels and no 5 min baseline; sniffing of the SW
mouse is annotated in one mouse; only one two-photon mouse has BALB/c intervals for its observation
session.